# Burgers Equation — Physics-Informed Neural Network

The **Burgers equation** is a fundamental nonlinear PDE that captures the interplay between advection and diffusion:

$$u_t + u \, u_x = \nu \, u_{xx}, \quad x \in [-1, 1], \quad t \in [0, 1]$$

with viscosity $\nu = 0.01/\pi$, initial condition $u(0, x) = -\sin(\pi x)$, and homogeneous Dirichlet boundary conditions $u(t, \pm 1) = 0$.

A **Physics-Informed Neural Network (PINN)** approximates the solution $u(t, x)$ by training a neural network whose loss function enforces:
1. Fit to the known initial condition
2. Fit to the boundary conditions
3. Satisfaction of the PDE at randomly sampled interior *collocation* points

All PDE derivatives are computed via PyTorch autograd — no finite differences, no labeled interior data.

> Reference: *Raissi, Perdikaris & Karniadakis — Physics-informed neural networks, J. Comput. Phys. 378 (2019)*

In [ ]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Network Architecture

The PINN is a fully-connected MLP that maps space-time coordinates $(t, x)$ to the scalar solution $\hat{u}(t, x)$.

**Design choices:**
- **9 layers** (1 input projection + 7 hidden + 1 output): deep enough to capture the sharp shock that forms near $t = 1$
- **40 neurons per layer**: matches the original Raissi et al. architecture
- **Tanh activations**: required to be $C^\infty$ so that autograd can compute $u_t$, $u_x$, $u_{xx}$ through the network

ReLU cannot be used here because $u_{xx}$ would be zero almost everywhere.

In [ ]:
class PINNs(nn.Module):
    """Fully-connected neural network that approximates u(t, x).

    Architecture: Linear(2→40) + 7×[Linear(40→40) + Tanh] + Linear(40→1).
    Tanh is used throughout because it is smooth (C∞), which is required for computing
    second-order PDE derivatives via autograd.
    """

    NUM_NEURONS = 40  # neurons per hidden layer (Raissi et al. default)
    NUM_LAYERS  = 9   # total layers: 1 input projection + 7 hidden + 1 output

    def __init__(self):
        super().__init__()
        layers = []

        # Input projection: (t, x) → hidden dimension
        layers.append(nn.Linear(2, self.NUM_NEURONS))
        layers.append(nn.Tanh())

        # Hidden layers
        for _ in range(self.NUM_LAYERS - 1):
            layers.append(nn.Linear(self.NUM_NEURONS, self.NUM_NEURONS))
            layers.append(nn.Tanh())

        # Output head: hidden → scalar u(t, x)
        layers.append(nn.Linear(self.NUM_NEURONS, 1))

        self.network = nn.Sequential(*layers)

    def forward(self, t, x):
        """Forward pass.

        Args:
            t: Time coordinates of shape (N, 1).
            x: Spatial coordinates of shape (N, 1).

        Returns:
            Predicted solution u(t, x) of shape (N, 1).
        """
        inputs = torch.cat([t, x], dim=1)  # concatenate along feature dim → (N, 2)
        return self.network(inputs)

## 2. PDE Residual

The Burgers residual quantifies how well the network satisfies the PDE at any point $(t, x)$:

$$f(t, x) \;=\; u_t + u \, u_x - \nu \, u_{xx}$$

A perfect solution has $f = 0$ everywhere. During training we minimise $\|f\|^2$ at $N_f = 10{,}000$ sampled collocation points.

**Autograd key:** `create_graph=True` builds a second computational graph through the derivative operation itself, so the residual loss $\|f\|^2$ is differentiable with respect to the network weights.

In [ ]:
def physics_residual(model, t, x):
    """Evaluate the Burgers PDE residual at collocation points.

    Computes f = u_t + u·u_x − ν·u_xx using automatic differentiation.
    A well-trained network produces f ≈ 0 everywhere in the domain.

    Args:
        model: The PINN model.
        t: Time coordinates of shape (N, 1). Will be detached and re-attached for grad.
        x: Spatial coordinates of shape (N, 1). Will be detached and re-attached for grad.

    Returns:
        PDE residual f of shape (N, 1).
    """
    nu = 0.01 / torch.pi  # viscosity coefficient

    # Detach from any prior graph and re-enable grad tracking for derivative computation
    t = t.clone().detach().requires_grad_(True)
    x = x.clone().detach().requires_grad_(True)

    u = model(t, x)  # shape (N, 1)

    # First-order temporal derivative: u_t
    u_t = torch.autograd.grad(
        u, t,
        grad_outputs=torch.ones_like(u),
        create_graph=True,   # needed so that loss.backward() flows through u_t
        retain_graph=True
    )[0]

    # First-order spatial derivative: u_x
    u_x = torch.autograd.grad(
        u, x,
        grad_outputs=torch.ones_like(u),
        create_graph=True,
        retain_graph=True
    )[0]

    # Second-order spatial derivative: u_xx = d(u_x)/dx
    u_xx = torch.autograd.grad(
        u_x, x,
        grad_outputs=torch.ones_like(u_x),
        create_graph=True,
        retain_graph=True
    )[0]

    # Burgers residual: f = u_t + u·u_x − ν·u_xx
    f = u_t + u * u_x - nu * u_xx

    return f

## 3. Training Data

The PINN is supervised with three types of labeled points — no interior solution data is needed:

| Set | Symbol | Count | Description |
|---|---|---|---|
| Initial condition | $\mathcal{T}_{IC}$ | 100 | $u(0, x) = -\sin(\pi x)$, $x$ uniform in $[-1, 1]$ |
| Left boundary | $\mathcal{T}_{BC}$ | 50 | $u(t, -1) = 0$, $t$ uniform in $[0, 1]$ |
| Right boundary | $\mathcal{T}_{BC}$ | 50 | $u(t, +1) = 0$, $t$ uniform in $[0, 1]$ |
| Collocation | $\mathcal{T}_f$ | 10 000 | Random interior points — **no labels**, only PDE enforcement |

In [ ]:
# ── Point counts ──────────────────────────────────────────────────────────────
N_ic = 100    # initial condition points at t = 0
N_bc = 100    # boundary condition points (50 per side)
N_f  = 10000  # collocation points (interior, no labels needed)

# ── Initial condition: t = 0, u(0, x) = -sin(πx) ─────────────────────────────
t_ic = torch.zeros(N_ic, 1)
x_ic = 2 * torch.rand(N_ic, 1) - 1          # x ~ Uniform[-1, 1]
u_ic = -torch.sin(torch.pi * x_ic)          # exact initial profile

# ── Dirichlet boundary conditions: u(t, ±1) = 0 ──────────────────────────────
t_bc_left  = torch.rand(N_bc // 2, 1)       # t ~ Uniform[0, 1]
x_bc_left  = -torch.ones(N_bc // 2, 1)      # x = -1
u_bc_left  = torch.zeros(N_bc // 2, 1)      # u = 0

t_bc_right = torch.rand(N_bc // 2, 1)
x_bc_right =  torch.ones(N_bc // 2, 1)      # x = +1
u_bc_right = torch.zeros(N_bc // 2, 1)      # u = 0

# Combine all labeled data into a single supervised batch
t_data = torch.cat([t_ic, t_bc_left, t_bc_right]).to(device)
x_data = torch.cat([x_ic, x_bc_left, x_bc_right]).to(device)
u_data = torch.cat([u_ic, u_bc_left, u_bc_right]).to(device)

# ── Collocation points: random interior samples for PDE enforcement ────────────
t_f = torch.rand(N_f, 1).to(device)
x_f = (2 * torch.rand(N_f, 1) - 1).to(device)

print(f"Labeled data points : {t_data.shape[0]}  (IC + BC)")
print(f"Collocation points  : {t_f.shape[0]}  (physics supervision only)")

## 4. Loss Function

The total loss combines three mean-squared error terms:

$$\mathcal{L} = \underbrace{\frac{1}{N_{IC}}\sum \|\hat{u} - u_{IC}\|^2}_{\text{initial condition}} + \underbrace{\frac{1}{N_{BC}}\sum \|\hat{u} - u_{BC}\|^2}_{\text{boundary conditions}} + \underbrace{\frac{1}{N_f}\sum \|f(t_i, x_i)\|^2}_{\text{PDE residual}}$$

In practice the IC and BC terms are concatenated into a single `mse_u`, and the residual term is `mse_f`.

In [ ]:
def compute_loss(model, t_data, x_data, u_data, t_f, x_f):
    """Compute the composite PINN loss.

    Args:
        model: The PINN model.
        t_data, x_data: Coordinates of labeled points (IC + BC), each (N, 1).
        u_data: Known solution values at labeled points, shape (N, 1).
        t_f, x_f: Collocation point coordinates for PDE enforcement, each (N_f, 1).

    Returns:
        total_loss: mse_u + mse_f (scalar).
        mse_u: Data loss — how well the network fits IC and BC.
        mse_f: Physics loss — how well the network satisfies the PDE.
    """
    # Data loss: initial condition + boundary conditions
    u_pred = model(t_data, x_data)
    mse_u = torch.mean((u_pred - u_data) ** 2)

    # Physics loss: PDE residual at collocation points
    f = physics_residual(model=model, t=t_f, x=x_f)
    mse_f = torch.mean(f ** 2)

    total_loss = mse_u + mse_f
    return total_loss, mse_u, mse_f

## 5. Two-Phase Training

Training follows the strategy from the original PINNs paper:

| Phase | Optimizer | Iterations | Purpose |
|---|---|---|---|
| 1 | Adam, lr = 1e-3 | 5 000 epochs | Rapid escape from random initialisation |
| 2 | L-BFGS, Strong Wolfe line search | up to 50 000 | High-precision convergence using curvature information |

Adam is good at navigating the rough early loss landscape; L-BFGS converges to machine precision once the network is already close to the solution. Switching too early keeps L-BFGS stuck in a poor local minimum.

In [ ]:
# ── Phase 1: Adam ─────────────────────────────────────────────────────────────
model     = PINNs().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("Phase 1: Adam")
for epoch in range(5000):
    optimizer.zero_grad()
    loss, mse_u, mse_f = compute_loss(
        model, t_data, x_data, u_data, t_f, x_f
    )
    loss.backward()
    optimizer.step()

    if epoch % 500 == 0:
        print(f"  Epoch {epoch:5d} | Loss: {loss.item():.6f} | "
              f"MSE_u: {mse_u.item():.6f} | MSE_f: {mse_f.item():.6f}")

# ── Phase 2: L-BFGS ───────────────────────────────────────────────────────────
# L-BFGS requires a closure that re-evaluates the loss and calls backward().
# strong_wolfe line search ensures sufficient decrease at each step.
optimizer_lbfgs = torch.optim.LBFGS(
    model.parameters(),
    lr=1.0,
    max_iter=50000,
    max_eval=50000,
    tolerance_grad=1e-7,
    tolerance_change=1e-9,
    history_size=50,
    line_search_fn="strong_wolfe"
)

iteration = [0]  # mutable counter accessible inside closure

def closure():
    optimizer_lbfgs.zero_grad()
    loss, mse_u, mse_f = compute_loss(
        model, t_data, x_data, u_data, t_f, x_f
    )
    loss.backward()

    iteration[0] += 1
    if iteration[0] % 500 == 0:
        print(f"  L-BFGS iter {iteration[0]:5d} | Loss: {loss.item():.6f} | "
              f"MSE_u: {mse_u.item():.6f} | MSE_f: {mse_f.item():.6f}")
    return loss

print("\nPhase 2: L-BFGS")
optimizer_lbfgs.step(closure=closure)
print(f"\nFinal loss: {closure().item():.6f}")

## 6. Exact Solution (Cole-Hopf Transformation)

The Burgers equation admits an exact solution via the **Cole-Hopf transformation**, which converts it into the linear heat equation. This lets us compute a ground-truth reference against which to measure the PINN's error.

The transformation: $u = -2\nu \frac{\phi_x}{\phi}$, where $\phi$ satisfies the heat equation.

In [ ]:
import numpy as np
from scipy.integrate import quad

def exact_burgers(t_val, x_val, nu=0.01 / np.pi):
    """Compute the exact Burgers solution via the Cole-Hopf transformation.

    Converts the nonlinear Burgers equation into the linear heat equation through
    the substitution u = -2ν (∂_x log φ), then evaluates φ via numerical quadrature.

    Args:
        t_val: Time value (scalar). Returns the IC directly if t_val == 0.
        x_val: Spatial value (scalar).
        nu: Viscosity coefficient, default 0.01/π.

    Returns:
        Exact scalar solution u(t_val, x_val).
    """
    if t_val == 0:
        return -np.sin(np.pi * x_val)

    # Heat kernel integrand weight: φ(ξ) = exp(−cos(πξ) / (2πν))
    def phi(xi):
        return np.exp(-np.cos(np.pi * xi) / (2 * np.pi * nu))

    # Numerator: ∫ sin(πξ) · φ(ξ) · G(x − ξ, t) dξ
    def integrand_num(xi):
        return np.sin(np.pi * xi) * phi(xi) * np.exp(-(x_val - xi) ** 2 / (4 * nu * t_val))

    # Denominator: ∫ φ(ξ) · G(x − ξ, t) dξ
    def integrand_den(xi):
        return phi(xi) * np.exp(-(x_val - xi) ** 2 / (4 * nu * t_val))

    num, _ = quad(integrand_num, -1, 1)
    den, _ = quad(integrand_den, -1, 1)

    return -num / (den + 1e-30)  # 1e-30 guards against exact-zero denominator

## 7. Validation & Visualisation

Evaluate the trained PINN on a $100 \times 100$ space-time grid and compare against the exact Cole-Hopf solution. Three panels are plotted:
1. **Exact solution** — ground truth
2. **PINN prediction** — what the network learned
3. **Absolute error** — where the network deviates

In [ ]:
import matplotlib.pyplot as plt

# ── Build evaluation grid ──────────────────────────────────────────────────────
t_grid = np.linspace(0, 1, 100)
x_grid = np.linspace(-1, 1, 100)
T, X   = np.meshgrid(t_grid, x_grid)

t_flat = torch.tensor(T.flatten(), dtype=torch.float32).unsqueeze(1).to(device)
x_flat = torch.tensor(X.flatten(), dtype=torch.float32).unsqueeze(1).to(device)

# ── PINN prediction ────────────────────────────────────────────────────────────
with torch.no_grad():
    u_pred = model(t_flat, x_flat).cpu().numpy().reshape(100, 100)

# ── Exact solution via Cole-Hopf (numerical quadrature) ───────────────────────
print("Computing exact solution (may take ~30 s)...")
u_exact = np.zeros_like(T)
for i in range(T.shape[0]):
    for j in range(T.shape[1]):
        u_exact[i, j] = exact_burgers(T[i, j], X[i, j])
print("Done.")

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

titles = ["Exact Solution", "PINN Prediction", "Absolute Error"]
data   = [u_exact, u_pred, np.abs(u_exact - u_pred)]
cmaps  = ["viridis", "viridis", "hot"]

for ax, title, d, cmap in zip(axes, titles, data, cmaps):
    c = ax.contourf(T, X, d, levels=50, cmap=cmap)
    ax.set_title(title)
    ax.set_xlabel("t")
    ax.set_ylabel("x")
    plt.colorbar(c, ax=ax)

plt.tight_layout()
plt.savefig("pinn_heat_equation.png", dpi=150)
plt.show()

max_err = np.max(np.abs(u_exact - u_pred))
print(f"Max absolute error: {max_err:.6f}")